In [5]:
import numpy as np
from PIL import Image
import os
import random

def extract_features(image_path):
    """Извлекает признаки изображения."""
    img = Image.open(image_path).convert('RGB')
    img_array = np.array(img) / 255.0  
    
    brightness = np.mean(img_array)
    green_ratio = np.mean(img_array[:, :, 1])  
    contrast = np.std(img_array)
    
    return np.array([brightness, green_ratio, contrast])

In [11]:
class DecisionNode:
    """Узел дерева решений."""
    def __init__(self, feature_idx=None, threshold=None, left=None, right=None, value=None):
        self.feature_idx = feature_idx  
        self.threshold = threshold      
        self.left = left                
        self.right = right              
        self.value = value              

class DecisionTreeClassifier:
    def __init__(self, max_depth=None):
        self.max_depth = max_depth
        self.root = None

    def fit(self, X, y):
        """Построение дерева."""
        self.root = self._build_tree(X, y, depth=0)

    def _build_tree(self, X, y, depth):
        """Рекурсивное построение дерева."""
        n_samples, n_features = X.shape
        n_classes = len(np.unique(y))

        # Критерии остановки
        if (self.max_depth is not None and depth >= self.max_depth) or (n_classes == 1):
            return DecisionNode(value=self._most_common_class(y))

        # Поиск лучшего разделения
        best_gini = float('inf')
        best_feature, best_threshold = None, None

        for feature_idx in range(n_features):
            thresholds = np.unique(X[:, feature_idx])
            for threshold in thresholds:
                left_mask = X[:, feature_idx] <= threshold
                gini = self._gini_impurity(y[left_mask], y[~left_mask])
                if gini < best_gini:
                    best_gini = gini
                    best_feature = feature_idx
                    best_threshold = threshold

        if best_gini == float('inf'):
            return DecisionNode(value=self._most_common_class(y))

        # Рекурсивное создание подузлов
        left_mask = X[:, best_feature] <= best_threshold
        left = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right = self._build_tree(X[~left_mask], y[~left_mask], depth + 1)

        return DecisionNode(feature_idx=best_feature, threshold=best_threshold, left=left, right=right)

    def _gini_impurity(self, left_y, right_y):
        """Расчёт примеси Джини для разделения."""
        n_left, n_right = len(left_y), len(right_y)
        n_total = n_left + n_right

        gini_left = 1 - sum((np.sum(left_y == c) / n_left) ** 2 for c in np.unique(left_y)) if n_left > 0 else 0
        gini_right = 1 - sum((np.sum(right_y == c) / n_right) ** 2 for c in np.unique(right_y)) if n_right > 0 else 0

        return (n_left / n_total) * gini_left + (n_right / n_total) * gini_right

    def _most_common_class(self, y):
        """Возвращает наиболее частый класс."""
        return Counter(y).most_common(1)[0][0]

    def predict(self, X):
        """Предсказание класса."""
        return np.array([self._predict_tree(x, self.root) for x in X])

    def _predict_tree(self, x, node):
        """Рекурсивное предсказание для одного образца."""
        if node.value is not None:
            return node.value
        if x[node.feature_idx] <= node.threshold:
            return self._predict_tree(x, node.left)
        else:
            return self._predict_tree(x, node.right)

In [12]:
from collections import Counter

def load_dataset(dataset_path):
    """Загружает изображения и метки классов."""
    X, y = [], []
    for class_name in ['forest', 'desert']:
        class_dir = os.path.join(dataset_path, class_name)
        for img_file in os.listdir(class_dir):
            img_path = os.path.join(class_dir, img_file)
            features = extract_features(img_path)
            X.append(features)
            y.append(class_name)
    return np.array(X), np.array(y)

In [14]:
# Путь к датасету
DATASET_PATH = "origins/task_2"

# Загрузка данных
X, y = load_dataset(DATASET_PATH)

# Преобразование меток в числовой формат (0 - forest, 1 - desert)
y = np.where(y == 'forest', 0, 1)

# Объединяем и перемешиваем
data = list(zip(X, y))
random.shuffle(data)  # Перемешиваем вместе
X_shuffled, y_shuffled = zip(*data)  # Разделяем обратно

# Разделение на обучающую и тестовую выборки
split_idx = int(0.8 * len(X_shuffled))
X_train, y_train = X[:split_idx], y[:split_idx]
X_test, y_test = X[split_idx:], y[split_idx:]

# Обучение дерева (максимальная глубина = 3)
tree = DecisionTreeClassifier(max_depth=5)
tree.fit(X_train, y_train)

# Предсказание для тестовых данных
y_pred = tree.predict(X_test)

# Оценка точности
accuracy = np.mean(y_pred == y_test)
print(f"Точность: {accuracy * 100:.2f}%")

Точность: 74.53%


In [15]:
img_path = 'origins/task_2/test/test_desert_1.jpg'
features = extract_features(img_path)

y_pred = tree.predict(np.array([features]))

print(y_pred[0])

0


In [16]:
img_path = 'origins/task_2/test/test_forest_2.jpg'
features = extract_features(img_path)

y_pred = tree.predict(np.array([features]))

print(y_pred[0])

0
